# TP 1 — Pipeline Iris : votre premier modèle

**Objectifs**

- Charger un jeu de données scikit-learn et l'explorer.
- Faire un split train/test stratifié reproductible.
- Entraîner et évaluer un k-NN.
- Examiner l'erreur classe par classe.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)

## Exercice 1 — Exploration

1. Chargez le jeu Iris avec `load_iris()`.
2. Affichez la shape de `X`, la liste des `feature_names`, des `target_names`, et le nombre d'exemples par classe.
3. Construisez un DataFrame pandas qui contient features + label en clair, et affichez les statistiques `describe()`.

<details>
<summary>💡 Coup de pouce — explorer un dataset sklearn</summary>

**🎯 But :** prendre en main un Bunch de scikit-learn et le convertir en DataFrame pour explorer rapidement.

**Charger Iris**

```python
from sklearn.datasets import load_iris
iris = load_iris()
```

`iris` est un **objet `Bunch`** (dict avec accès par attribut). Il contient :
- `iris.data` : tableau NumPy `(150, 4)` = 150 fleurs × 4 mesures (sépales/pétales en cm)
- `iris.target` : tableau `(150,)` d'entiers 0/1/2 = espèce
- `iris.feature_names` : `['sepal length', 'sepal width', 'petal length', 'petal width']`
- `iris.target_names` : `['setosa', 'versicolor', 'virginica']`

**Conversion en DataFrame pandas**

```python
import pandas as pd
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['label'] = [iris.target_names[i] for i in iris.target]
```

Le DataFrame mélange 4 colonnes numériques + 1 colonne catégorielle. Très pratique pour `df.describe()`, `df.head()`, et le countplot.

**Compter les classes**

```python
df['label'].value_counts()
```

→ devrait montrer **50 / 50 / 50** : dataset parfaitement équilibré (typique des datasets « jouets »).

**Visualisations utiles pour démarrer**
- `df.hist(figsize=(10, 6))` : distribution de chaque feature
- `pd.plotting.scatter_matrix(df, c=iris.target, figsize=(10, 10))` : nuages 2 à 2

Vous verrez tout de suite que **setosa** est facilement séparable, **versicolor/virginica** se chevauchent → ce qui va définir la difficulté du problème.

</details>

In [2]:
# TODO


## Exercice 2 — Split train/test

Séparez les données en train (80 %) et test (20 %), de façon **stratifiée** (les proportions de classes sont préservées), avec `random_state=0`. Vérifiez visuellement que les proportions sont bien respectées.

<details>
<summary>💡 Coup de pouce — split train/test stratifié</summary>

**🎯 But :** séparer les données en train/test en **conservant les proportions de classes**.

**Le split standard**

```python
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target,
    test_size=0.2,
    stratify=iris.target,    # ← le mot-clé magique
    random_state=0
)
```

**Pourquoi `stratify=y` ?**

Sans stratification, le split est purement aléatoire. Sur 150 échantillons avec 3 classes équilibrées, on pourrait par malchance avoir :
- Train : 50 setosa, 45 versicolor, 25 virginica
- Test : 0 setosa, 5 versicolor, 25 virginica

→ Le classifieur ne verrait quasiment pas de virginica en train et serait testé surtout dessus → résultats biaisés.

Avec `stratify=y`, sklearn s'assure que **chaque sous-ensemble a la même répartition de classes** que les données originales.

**Pourquoi `random_state=0` ?**

Garantit la **reproductibilité**. Sans ça, chaque run de votre code donne un split différent → les scores changent → impossible de comparer deux essais. En production : on retire pour avoir une vraie variabilité. En cours/debug : on fixe toujours.

**Vérification**

```python
import numpy as np
np.unique(y_train, return_counts=True)   # devrait donner ~40, 40, 40
np.unique(y_test, return_counts=True)    # devrait donner ~10, 10, 10
```

→ les proportions train et test sont les mêmes que le dataset complet (50/50/50 ratio).

</details>

In [3]:
# TODO


## Exercice 3 — k-NN

1. Entraînez un `KNeighborsClassifier` avec `n_neighbors=5`.
2. Calculez l'accuracy sur train et sur test. Commentez l'éventuel écart.
3. Affichez le `classification_report` sur le test.
4. Affichez la matrice de confusion avec `ConfusionMatrixDisplay`.

<details>
<summary>💡 Coup de pouce — k-NN et évaluation</summary>

**🎯 But :** entraîner un classifieur k plus proches voisins et mesurer ses performances avec accuracy + matrice de confusion.

**Entraîner k-NN**

```python
from sklearn.neighbors import KNeighborsClassifier
clf = KNeighborsClassifier(n_neighbors=5)
clf.fit(X_train, y_train)
```

⚠️ k-NN est **non-paramétrique** : `.fit()` ne fait que **stocker** les données d'entraînement. Tout le calcul a lieu au moment de la prédiction (chercher les k voisins les plus proches).

**Mesurer l'accuracy**

Deux moyens équivalents :

```python
from sklearn.metrics import accuracy_score
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
# ou directement :
acc = clf.score(X_test, y_test)
```

Sur Iris avec `k=5`, attendez-vous à **0.93-1.0**. C'est un dataset facile.

**Matrice de confusion**

```python
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(
    clf, X_test, y_test, display_labels=iris.target_names
)
```

Lecture : ligne = vraie classe, colonne = classe prédite. Les valeurs hors diagonale = erreurs. Sur Iris, les seules confusions seront entre **versicolor** et **virginica** (le doublon difficile vu en Exo 1).

**Détecter l'overfit**

```python
acc_train = clf.score(X_train, y_train)
acc_test  = clf.score(X_test, y_test)
print(f"Train: {acc_train:.3f}  |  Test: {acc_test:.3f}")
```

- **Écart faible** (ex. 0.97 train vs 0.95 test) → modèle bien régularisé.
- **Écart énorme** (ex. 1.00 train vs 0.70 test) → overfit : le modèle a mémorisé le train mais ne généralise pas.

</details>

In [4]:
# TODO


## Exercice 4 — Sensibilité à `k`

Pour `k` de 1 à 20, entraînez un k-NN, mesurez l'accuracy sur train et sur test, et tracez les deux courbes.

Quel `k` est le meilleur sur le test ? Que se passe-t-il pour `k=1` ?

<details>
<summary>💡 Coup de pouce — sensibilité de k-NN au choix de k</summary>

**🎯 But :** comprendre visuellement comment `k` influence le compromis biais/variance.

**Boucle sur les valeurs de k**

```python
ks = range(1, 21)
train_scores, test_scores = [], []
for k in ks:
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    test_scores.append(clf.score(X_test, y_test))
```

**Tracer**

```python
plt.plot(ks, train_scores, marker='o', label='train')
plt.plot(ks, test_scores,  marker='s', label='test')
plt.xlabel('k'); plt.ylabel('accuracy'); plt.legend()
```

**Ce que vous devriez voir**

- **k=1** : train accuracy = **1.0** (chaque point d'entraînement est son propre voisin) ; test accuracy souvent **plus faible** (modèle bruité, sensible aux outliers).
- **k=5-10** : sweet spot, train et test proches.
- **k=20+** : sous-ajustement, le modèle moyenne trop large, perd la finesse locale.

C'est l'illustration classique du **compromis biais-variance** :
- Petit `k` = **forte variance** (overfit), faible biais
- Grand `k` = **fort biais** (underfit), faible variance

**Choisir le bon k en production**

Pas de magie, mais 3 règles :
1. Toujours **impair** pour les classifications à 2 classes (évite les égalités de vote).
2. Souvent une valeur autour de `sqrt(N)` où `N` = taille du train.
3. Validation croisée (vu en TP9) pour automatiser le choix.

</details>

In [5]:
# TODO
